# Superoperator Representations

This document summarizes our conventions for the different superoperator representations. We show how to apply the channels to states in these representations and how to convert channels between a subset of representations. By combining these conversion methods you can convert between any of the channel representations.

This document is **not** intended to be a tutorial or a comprehensive review. At the bottom of the document there is a list of references with more information. This document was influenced by [IGST] and we recommend reading [GRAPTN] to gain deeper understanding (see also [QN], [SVDMAT], [MATQO], [DUAL] listed at the bottom of this document). Additionally, these references explain, for example, how to determine if a channel is unital or completely positive in the different representations.

This notebook is adapted from the [forest-benchmarking](https://github.com/rigetti/forest-benchmarking) documentation on superoperator representations, with added code examples demonstrating Quax's implementation.

In [ ]:
import jax.numpy as jnp
import quax as qx
from quax.gates import I, Z, H

## `vec` and `unvec`

Consider an $m\times m$ matrix

$$ A = [a_{ij}] = \begin{pmatrix}  
a_{11} & a_{12} & \ldots & a_{1m} \\
a_{21} & a_{22} & \ldots & a_{2m}\\ 
\vdots &   & \ddots & \vdots\\ 
a_{m1} & a_{m2} & \ldots & a_{mm} 
\end{pmatrix}$$

where $i$ is a row and $j$ is a column index.

We define `vec` to be column stacking

$$ |A\rangle \rangle :={\rm vec}(A) = (a_{11},a_{21},\ldots,a_{m1},a_{12},\ldots,a_{mm})^T \quad (1) $$

where $T$ denotes a transpose. Clearly an inverse operation, `unvec` can be defined so that 

$$ {\rm unvec}\big ( {\rm vec}(A) \big ) = A.$$

Of course `unvec()` generally depends on the dimensions of $A$, which are not recoverable from `vec(A)`. We often focus on square $A$, but for generality, we require the dimensions for $A$, defaulting to the square root of the dimension of `vec(A)`. Column stacking corresponds to how matrices are stored in memory for column major storage conventions.

Similarly we can define a row vectorization to be row stacking $\mathrm{vec}_r(A) = (a_{11}, a_{12}, \ldots, a_{1m}, a_{21},\ldots, a_{mm})^T$. Note that $\mathrm{vec}(A) = \mathrm{vec}_r(A^T)$. In any case we will **not** use this row convention.

### Matrix multiplication in vectorized form

For matrices $A,B,C$ 

$$\begin{align}
{\rm vec}(ABC) = (C^T\otimes A) {\rm vec}(B), \quad (2)
\end{align}$$

which is sometimes called Roth's lemma.

Eq. 2 is useful in representing quantum operations on mixed quantum states. For example consider 

$$ \rho' = U \rho U^\dagger.$$

We can use Eq. 1 to write this as

$$ {\rm vec}(\rho') = \{(U^\dagger)^T \otimes U \} {\rm vec}(\rho)
= (U^*\otimes U) |\rho\rangle\rangle$$

so 

$$ |\rho'\rangle \rangle = \mathcal U |\rho\rangle\rangle,
$$

where $\mathcal U = U^*\otimes U$. The nice thing about this is the operator (the state) has become a vector and the superoperator (the left right action of $U$) has become an operator.

Some other useful results related to vectorization are

$ {\rm vec}([A,X])= (I\otimes A - A^T\otimes I) {\rm vec}(X)$

${\rm vec}(ABC) = (I\otimes AB) {\rm vec}( C ) = (C^T B^T\otimes I) {\rm vec}(A)$

${\rm vec}(AB) = (I\otimes A) {\rm vec}(B) = (B^T\otimes I) {\rm vec}(A)$

## Matrix operations on Bipartite matrices: Reshuffling, SWAP, and transposition

This section is based on the Wood et al. presentation in [GRAPTN].

As motivation for this section consider the Kraus representation theorem. It shows that a quantum channel can be represented as a partial trace over a unitary operation on a larger Hilbert space. Actually the unitary is on a bipartite Hilbert space.

When representing quantum channels one insight is used many times.

Consider two Hilbert spaces $\mathbb H_A$ and $\mathbb H_B$ with dimensions $d_A$ and $d_B$ respectively. An abstract quantum process matrix $\mathcal Q$ lives in the combined (bipartite) space of $\mathbb H_A \otimes \mathbb H_B$ so $\mathcal Q$ is a $d_A^2\times d_B^2$ matrix.

We can represent the process as a tensor with components

$$\mathcal Q_{m,\mu;n,\nu} = \langle m, \mu |\mathcal Q |n,\nu \rangle $$ 

where $|n,\nu\rangle = |n\rangle \otimes |\nu\rangle$, $m,n\in \{0,\ldots, d_A-1\}$, $\mu,\nu\in \{0,\ldots, d_B-1\}$ and all vectors are in the standard basis.

With respect to these indices some useful operations are [GRAPTN]:
 
- Transpose $T$: $\mathcal Q_{m,\mu;n,\nu} \mapsto \mathcal Q_{n,\nu;m,\mu}$  
- SWAP: $\mathcal Q_{m,\mu;n,\nu} \mapsto \mathcal Q_{\mu,m;\nu,n}$  
- Row-reshuffling $R_r$: $\mathcal Q_{m,\mu;n,\nu} \mapsto \mathcal Q_{m,n;\mu,\nu}$  
- Col-reshuffling $R$: $\mathcal Q_{m,\mu;n,\nu} \mapsto \mathcal Q_{\nu,\mu;n,m}$

The importance of understanding reshuffling can be understood as understanding the relationship between 

$${\rm vec}(G)\otimes {\rm vec}(\Gamma) \quad {\rm and} \quad  {\rm vec}(G\otimes\Gamma)$$

where $G$ and $\Gamma$ are matrices that act on $\mathbb H_A$ and $\mathbb H_B$ respectively, as explained in [VECQO].

### A note on numerical implementations

Most linear algebra (or tensor) libraries have the ability to `reshape` a matrix and `swapaxes` (or sometimes it is called `permute_dims`). 

If you are trying to reshuffle indices usually the first job is to write your matrix in tensor form. This requires reshaping a $d_A^2\times d_B^2$ matrix into a $d_A\times d_A\times d_B \times d_B$ tensor. Next you `permute_dims` or `swapaxes`. Often $d_A = d_B$ so we `reshape` to a matrix that has the same dimensions as the original $d_A^2\times d_A^2$ matrix.

**Quax note:** This is exactly how Quax stores superoperators internally -- in tensor format -- so reshuffling operations are just axis permutations on the underlying data.

## The $n$-qubit Pauli basis

The $n$-qubit Pauli basis is denoted $\mathcal P^{\otimes n}$ where $\mathcal P = \{ I, X, Y, Z \}$ are the usual Pauli matrices. It is an operator basis for the $d = 2^n$ dimensional Hilbert space and there are $d^2 = 4^n$ operators in $\mathcal P^{\otimes n}$. If one divides all the operators by $\sqrt{d}$ the basis is orthonormal with respect to the Hilbert-Schmidt inner product.
 
It is often convenient to index the $d^2$ operators with a single label, e.g. $P_1=I^{\otimes n},\, \ldots,\, P_{d^2}= Z^{\otimes n}$  (or $P_0=I^{\otimes n}$ if you like zero indexing). In any case, as these operators are Hermitian and unitary they obey $P_i^2=I^{\otimes n}$.
 
To be explicit, for two qubits $d=4$ and we have 16 operators e.g. $\{II, IX, IY, IZ, XI, XX, XY, \ldots, ZZ\}$ where $II$ should be interpreted as $I\otimes I$ etc. The single index would be $\{P_1, P_2, P_3, P_4, P_5, P_6, P_7, \ldots, P_{16}\}$.

In Quax, for qubit systems this is the standard Pauli basis. For general qudit systems, Quax uses the **Hermitian Weyl basis** -- a generalization of the Pauli basis to arbitrary dimensions.

In [ ]:
# The single-qubit Pauli basis in Quax
pauli_basis_1q = qx.hermitian_weyl_basis(qudit_dim=2)
print(f"Single-qubit basis: {pauli_basis_1q.matrix.shape[0]} operators")
for label, op in zip(qx.hermitian_weyl_basis_labels(qudit_dim=2), pauli_basis_1q.matrix):
    print(f"  {label}:")
    print(f"    {jnp.round(op, 3)}")

# Two-qubit Pauli basis
pauli_basis_2q = qx.n_qudit_herm_basis(dims=(2, 2))
print(f"\nTwo-qubit basis: {pauli_basis_2q.matrix.shape[0]} operators")

## Quantum channels in the Kraus decomposition (operator-sum representation)

A completely positive map on the state $\rho$ can be written using a set of Kraus operators $\{ M_k \}$ as

$$\rho' =\mathcal E (\rho) = \sum_{k=1}^N M_k \rho M_k^\dagger, $$

where $\rho'$ is the state at the output of the channel.

If $\sum_k M_k^\dagger M_k= I$ the map is trace preserving. It turns out that $N\le d^2$ where $d$ is the Hilbert space dimension e.g. $d=2^n$ for $n$ qubits. Kraus operators are not necessarily unique -- sometimes there is a unitary degree of freedom in the Kraus representation.

In [ ]:
# Example: depolarizing channel Kraus operators
p = 0.1
kraus_ops = qx.depolarizing_operators(jnp.array(p))
print(f"Depolarizing channel (p={p}): {kraus_ops.matrix.shape[0]} Kraus operators")
for i, K in enumerate(kraus_ops.matrix):
    print(f"  M_{i} =\n    {jnp.round(K, 4)}")

# Verify trace preservation: sum_k M_k^dag M_k = I
tp_check = sum(K.conj().T @ K for K in kraus_ops.matrix)
print(f"\nTrace preservation check (should be I):\n  {jnp.round(tp_check, 6)}")

## Kraus to $\chi$ matrix (aka chi or process matrix)

We choose to represent the $\chi$ matrix in the Pauli basis. So we expand each of the Kraus operators in the $n$-qubit Pauli basis 

$$M_k = \sum^{d^2}_{j=1}c_{kj}\,P_j$$

where $P_j \in \mathcal P ^{\otimes n}$.

Now the channel $\mathcal E$ can be written as

$$\mathcal E (\rho) = \sum_{i,j=1}^{d^2} \chi_{i,j} P_i\rho P_j ,$$

where 

$$\chi_{i,j} = \sum_k c_{k,i} c_{k,j}^*$$ 

is an element of the process matrix $\chi$ of size $d^2 \times d^2$. If the channel is CP the $\chi$ matrix is Hermitian and positive semidefinite. 

The $\chi$ matrix can be related to the (yet to be defined) Choi matrix via a change of basis. Typically the Choi matrix is defined in the computational basis, while the $\chi$ matrix uses the Pauli basis. Moreover, they may have different normalization conventions.

In this light, after reviewing the Kraus to Choi conversion it is simple to see that the above is equivalent to first defining 

$$|c_{k}\rangle\rangle = U_{c2p}{\rm vec}(M_k)$$

then

$$\chi = \sum_k |c_{k}\rangle\rangle \langle\langle c_k|.$$

## Kraus to Pauli-Liouville matrix (Pauli transfer matrix)

We begin by defining the Pauli vector representation of the state $\rho$ 

$$ |\rho \rangle \rangle = \sum_j c_j |P_j\rangle \rangle$$

where $P_j \in \mathcal P^{\otimes n}$ and $c_j = (1/d) \langle\langle P_j|\rho \rangle\rangle$.

The Pauli-Liouville or Pauli transfer matrix representation of the channel $\mathcal E$ is denoted by $R_{\mathcal E}$. The matrix elements are

$$(R_{\mathcal E})_{i,j} = \frac 1 d {\rm Tr}[P_i \mathcal E(P_j)].$$

Trace preservation implies $(R_{\mathcal E})_{0,j} = \delta_{0,j}$, i.e. the first row is one and all zeros. Unitality implies $(R_{\mathcal E})_{i,0} = \delta_{i,0}$, the first column is one and all zeros.

In this representation the channel is applied to the state by multiplication

$$|\rho' \rangle \rangle = R_{\mathcal E} |\rho \rangle \rangle.$$

## Kraus to Superoperator (Liouville)

We already saw an example of this in the section on `vec`-ing. There we re-packaged conjugation by unitary evolution into the action of a matrix on a vec'd density operator. Unitary evolution is simply the case of a single Kraus operator, so we generalize this by taking a sum over all Kraus operators.  

Consider the set of Kraus operators $\{ M_k \}$. The corresponding quantum operation is $\mathcal E (\rho) = \sum_k M_k \rho M_k^\dagger$.

Using the vec operator (see Eq. 1) this implies a superoperator

$$\mathcal E = \sum_k (M_k^\dagger)^T \otimes M_k = \sum_k M_k^* \otimes M_k,$$

which acts as $\mathcal E |\rho\rangle \rangle$ using Equation 2.

**Note:** In quantum information a superoperator is an abstract concept. The object above is a concrete representation of the abstract concept in a particular basis. In the NMR community this particular construction is called the Liouville representation. The Pauli-Liouville representation is attained from the Liouville representation by a change of basis, so the similarity in naming makes sense.

## Kraus to Choi

Define $|\eta \rangle = \frac{1}{\sqrt{d}}\sum_{i=0}^{d-1}|i,i \rangle$

One can show that 

$$|A\rangle \rangle = {\rm vec}(A) = \sqrt{d} (I\otimes A) |\eta\rangle.$$

The Choi state is 

$$\begin{align}
\mathcal C &= I\otimes \mathcal E (|\eta \rangle \langle \eta|) \\
&=\sum_i (I \otimes M_i) |\eta \rangle \langle \eta  | ( I \otimes M_i^\dagger)\\
& = \frac{1}{d} \sum_i {\rm vec}(M_i)  {\rm vec} (M_i) ^\dagger \\
& = \frac{1}{d} \sum_i |M_i\rangle \rangle \langle\langle M_i |. 
\end{align}$$

An often quoted equivalent expression is

$$\begin{align}
\mathcal C &= I\otimes \mathcal E (|\eta \rangle \langle \eta|) \\
&=\sum_{ij} |i\rangle \langle j| \otimes  \mathcal E (|i \rangle \langle j | ).
\end{align}$$

## Conversion summary

### Superoperator to Pauli-Liouville matrix

The standard basis on $n$ qubits is called the computational basis. To convert between a superoperator and the Pauli-Liouville matrix representation we need to do a change of basis from the computational basis to the Pauli basis. This is achieved by the unitary

$$ U_{c2p}= \sum_{k=1}|c_k\rangle\langle\langle P_k|.$$

Then we have

$$ R_{\mathcal E} =  U_{c2p} \mathcal E U_{c2p}^\dagger.$$

### Superoperator to Choi

The conversion from the superoperator to a Choi matrix $\mathcal C$ is simply a (column) reshuffling operation

$$ \mathcal C = R(\mathcal E).$$

It turns out that $\mathcal E = R(\mathcal C)$ which means that $\mathcal E= R(R(\mathcal E))$.

### Pauli-Liouville to Choi

We obtain the normalized Choi matrix using the expression

$$ \rho_{\mathcal E} = \frac{1}{d^2}\sum_{i,j=1}^{d^2} (R_{\mathcal E})_{i,j}  \, P_j^T \otimes P_i.$$

### Choi to Kraus

Given the Choi matrix $\mathcal C$ we find its eigenvalues $\{\lambda_i\}$ and vectors $\{|M_i\rangle\rangle \}$. Then the Kraus operators are

$$ M_i = \sqrt{\lambda_i}\, {\rm unvec}\big (|M_i\rangle\rangle\big),$$

### Choi to Pauli-Liouville

First we normalize the Choi representation

$$\rho_{\mathcal E}=\frac 1 d \mathcal C$$

Then

$$(R_{\mathcal E})_{i,j} ={\rm Tr}[ \rho_{\mathcal E} \, P_j^T \otimes P_i].$$

## Conversions in Quax

Quax implements all of the above conversions. Let's demonstrate them with the Hadamard gate and a depolarizing channel.

In [ ]:
# Start from a unitary: the Hadamard gate
print("=== Hadamard gate ===")
print(f"H =\n{jnp.round(H.matrix, 4)}\n")

# Convert to all superoperator representations
H_superop = qx.unitary_to_superop(H)
H_choi = qx.unitary_to_choi(H)
H_pl = qx.unitary_to_pauli_liouville(H)
H_kraus = qx.unitary_to_kraus_map(H)

print(f"SuperOp (H* x H):\n{jnp.round(jnp.real(H_superop.matrix), 4)}\n")
print(f"Choi:\n{jnp.round(jnp.real(H_choi.matrix), 4)}\n")
print(f"Pauli-Liouville:\n{jnp.round(jnp.real(H_pl.matrix), 4)}\n")
print(f"Kraus operators: {H_kraus.matrix.shape[0]} operator(s)")
print(f"  M_0 =\n    {jnp.round(H_kraus.matrix[0], 4)}")

In [ ]:
# Round-trip conversions: verify consistency
print("=== Round-trip conversion checks ===")

# SuperOp -> Choi -> SuperOp
H_superop_rt = qx.choi_to_superop(qx.superop_to_choi(H_superop))
print(f"SuperOp -> Choi -> SuperOp matches: {jnp.allclose(H_superop.matrix, H_superop_rt.matrix, atol=1e-10)}")

# SuperOp -> PauliLiouville -> SuperOp
H_superop_rt2 = qx.pauli_liouville_to_superop(qx.superop_to_pauli_liouville(H_superop))
print(f"SuperOp -> PL -> SuperOp matches:    {jnp.allclose(H_superop.matrix, H_superop_rt2.matrix, atol=1e-10)}")

# Choi -> Kraus -> Choi
H_choi_rt = qx.kraus_to_choi(qx.choi_to_kraus(H_choi))
print(f"Choi -> Kraus -> Choi matches:       {jnp.allclose(H_choi.matrix, H_choi_rt.matrix, atol=1e-10)}")

# PL -> Choi -> PL
H_pl_rt = qx.choi_to_pauli_liouville(qx.pauli_liouville_to_choi(H_pl))
print(f"PL -> Choi -> PL matches:            {jnp.allclose(H_pl.matrix, H_pl_rt.matrix, atol=1e-10)}")

## Examples: One qubit channels

Some observations:  

* The Choi matrix of a unitary process always has rank 1.
* The superoperator / Liouville representation of a unitary process is always full rank.
* The eigenvalues of a Choi matrix give you an upper bound to the probability a particular (canonical) Kraus operator will occur.
* The $\chi$ matrix (in the Pauli basis) is very convenient for computing the result of Pauli twirling or Clifford twirling the corresponding process.

### Unitary Channels

As an example we look at $R_z(\theta) = \exp(-i \theta Z/2)$ and $H$. The Hadamard transforms $X$ and $Z$ to each other:

$$H Z H^\dagger = X, \qquad H X H^\dagger = Z$$

**Pauli-Liouville matrix:**

$$R_{R_z(\theta)}=
\begin{pmatrix}  
1 & 0 & 0 & 0 \\
0 & \cos(\theta) & -\sin(\theta) & 0 \\ 
0 & \sin(\theta) & \cos(\theta) & 0 \\ 
0 & 0 & 0 & 1 
\end{pmatrix}, \qquad
R_{H}=
\begin{pmatrix}  
1 & 0 & 0 & 0 \\
0 & 0 & 0 & 1 \\ 
0 & 0 & -1 & 0 \\ 
0 & 1 & 0 & 0
\end{pmatrix}$$

**Superoperator:**

$$ \mathcal R_z(\theta) = R_z^*\otimes R_z=
\begin{pmatrix}  
1 & 0 & 0 & 0 \\
0 & e^{i\theta} & 0 & 0\\ 
0 & 0  & e^{-i\theta} & 0\\ 
0 & 0 & 0 & 1 
\end{pmatrix}, \qquad
\mathcal H = H^*\otimes H=\frac 1 2
\begin{pmatrix}  
1 & 1 & 1 & 1 \\
1 & -1 & 1 & -1\\ 
1 & 1  & -1 &-1\\ 
1 & -1 & -1 & 1 
\end{pmatrix} 
$$

**Choi:**

$$\mathcal C_{R_z} = \frac 1 2
\begin{pmatrix}  
1 & 0 & 0 & e^{-i\theta} \\
0 & 0 & 0 & 0\\ 
0 & 0 & 0 & 0\\ 
e^{i\theta} & 0 & 0 & 1 
\end{pmatrix}, \qquad
\mathcal C_H = \frac 1 2
\begin{pmatrix}  
1  & 1  &  1 & -1 \\
1  & 1  &  1 & -1\\ 
1  & 1  &  1 & -1\\ 
-1 & -1 & -1 &  1 
\end{pmatrix}$$

In [ ]:
# Verify the representations match the formulas above
from quax.gates import RZ

theta = jnp.pi / 3
rz = RZ(theta)

print("=== RZ(pi/3) representations ===")
print(f"Pauli-Liouville:\n{jnp.round(jnp.real(qx.unitary_to_pauli_liouville(rz).matrix), 4)}\n")
print(f"SuperOp:\n{jnp.round(qx.unitary_to_superop(rz).matrix, 4)}\n")
print(f"Choi:\n{jnp.round(qx.unitary_to_choi(rz).matrix, 4)}\n")

print("=== Hadamard representations ===")
print(f"Pauli-Liouville:\n{jnp.round(jnp.real(qx.unitary_to_pauli_liouville(H).matrix), 4)}\n")
print(f"SuperOp:\n{jnp.round(jnp.real(qx.unitary_to_superop(H).matrix), 4)}\n")
print(f"Choi:\n{jnp.round(jnp.real(qx.unitary_to_choi(H).matrix), 4)}")

### Pauli Channels

Pauli channels are diagonal in two representations and have the depolarizing channel as a special case.

$$\mathcal E(\rho) = (1-p_x-p_y-p_z) I \rho I + p_x X\rho X + p_y Y \rho Y + p_z Z \rho Z$$

The depolarizing channel is the special case $p_x = p_y = p_z = p/4$.

**Pauli-Liouville matrix:**

$$R_{\mathcal E}=
\begin{pmatrix}  
1 & 0 & 0 & 0 \\
0 & 1-2(p_y+p_z) & 0 & 0 \\ 
0 & 0 & 1-2(p_x+p_z) & 0 \\ 
0 & 0 & 0 & 1-2(p_x+p_y) 
\end{pmatrix}$$

In [ ]:
# Depolarizing channel in all representations
p = 0.1
print(f"=== Depolarizing channel (p={p}) ===")

kraus = qx.depolarizing_operators(jnp.array(p))
print(f"Kraus operators: {kraus.matrix.shape[0]}")
for i, K in enumerate(kraus.matrix):
    print(f"  M_{i} = {jnp.round(jnp.real(K), 4)}")

superop = qx.kraus_to_superop(kraus)
choi = qx.kraus_to_choi(kraus)
pl = qx.kraus_to_pauli_liouville(kraus)

print(f"\nPauli-Liouville (diagonal for Pauli channels):\n{jnp.round(jnp.real(pl.matrix), 4)}")
print(f"\nChoi:\n{jnp.round(jnp.real(choi.matrix), 4)}")
print(f"\nSuperOp:\n{jnp.round(jnp.real(superop.matrix), 4)}")

### Amplitude Damping ($T_1$ channel)

Amplitude damping is an energy dissipation process. It is an example of a **non-unital** channel: $\mathcal E_{AD} (I) \neq I$.

**Kraus operators:**

$$M_0 = \begin{pmatrix} 1 & 0 \\ 0 & \sqrt{1-\gamma} \end{pmatrix}, \qquad
M_1 = \begin{pmatrix} 0 & \sqrt{\gamma} \\ 0 & 0 \end{pmatrix}$$

**Pauli-Liouville matrix:**

$$R_{AD}=
\begin{pmatrix}  
1 & 0 & 0 & 0 \\
0 & \sqrt{1-\gamma} & 0 & 0 \\ 
0 & 0 & \sqrt{1-\gamma} & 0 \\ 
\gamma & 0 & 0 & 1-\gamma 
\end{pmatrix}$$

Note the non-zero entry in the first column -- the signature of a non-unital channel.

In [ ]:
# Amplitude damping / relaxation channel
gamma = 0.3
kraus_ad = qx.relaxation_operators(jnp.array(gamma))
print(f"=== Amplitude damping (gamma={gamma}) ===")
print("Kraus operators:")
for i, K in enumerate(kraus_ad.matrix):
    print(f"  M_{i} =\n    {jnp.round(jnp.real(K), 4)}")

pl_ad = qx.kraus_to_pauli_liouville(kraus_ad)
choi_ad = qx.kraus_to_choi(kraus_ad)

print(f"\nPauli-Liouville (note non-zero first column = non-unital):\n{jnp.round(jnp.real(pl_ad.matrix), 4)}")
print(f"\nChoi:\n{jnp.round(jnp.real(choi_ad.matrix), 4)}")

# Verify non-unitality
print(f"\nFirst column of PL: {jnp.round(jnp.real(pl_ad.matrix[:, 0]), 4)}")
print(f"Channel is unital: {jnp.allclose(jnp.real(pl_ad.matrix[1:, 0]), 0.0, atol=1e-10)}")

## Examples: Two qubit channels

We consider two channels:

(1) A unitary channel: $\mathcal U_{IZ}(\rho) = (I\otimes Z) \rho (I\otimes Z)^\dagger$

(2) A dephasing channel: $\mathcal E_{IZ}(\rho) = (1-p)II \rho II + p\, IZ \rho IZ$

In [ ]:
# Two-qubit IZ unitary channel
IZ = I | Z
S_IZ = qx.unitary_to_superop(IZ)
print("=== IZ unitary channel ===")
print(f"SuperOp diagonal: {jnp.round(jnp.real(jnp.diag(S_IZ.matrix)), 1)}")

# Two-qubit dephasing: tensor product of single-qubit dephasing channels
p_deph = 0.05
deph_1q = qx.dephasing_operators(jnp.array(p_deph))
S_deph_1q = qx.kraus_to_superop(deph_1q)
S_deph_2q = S_deph_1q | S_deph_1q
print(f"\n=== 2Q dephasing channel (p={p_deph}) ===")
print(f"SuperOp diagonal: {jnp.round(jnp.real(jnp.diag(S_deph_2q.matrix)), 4)}")

## CPTP verification in Quax

Quax can verify that a channel is completely positive and trace preserving (CPTP) in any representation.

In [ ]:
p = 0.1
dep_kraus = qx.depolarizing_operators(jnp.array(p))
dep_choi = qx.kraus_to_choi(dep_kraus)
dep_pl = qx.kraus_to_pauli_liouville(dep_kraus)
dep_superop = qx.kraus_to_superop(dep_kraus)

print("CPTP checks for depolarizing channel:")
print(f"  Kraus is CPTP:    {qx.is_cptp(dep_kraus)}")
print(f"  Choi is CPTP:     {qx.is_cptp(dep_choi)}")
print(f"  PL is CPTP:       {qx.is_cptp(dep_pl)}")
print(f"  SuperOp is CPTP:  {qx.is_cptp(dep_superop)}")

# The Choi matrix of a CPTP map is positive semidefinite
eigvals = jnp.linalg.eigvalsh(dep_choi.matrix)
print(f"\nChoi eigenvalues: {jnp.round(eigvals, 6)}")
print(f"All non-negative: {jnp.all(eigvals >= -1e-10)}")

## References

- [IGST] Introduction to Quantum Gate Set Tomography. Greenbaum. arXiv:1509.02921 (2015). https://arxiv.org/abs/1509.02921

- [QN] Quantum Nescimus. Harper. PhD thesis University of Sydney (2018). https://ses.library.usyd.edu.au/handle/2123/17896

- [GRAPTN] Tensor networks and graphical calculus for open quantum systems. Wood et al. Quant. Inf. Comp. 15, 0579-0811 (2015). https://arxiv.org/abs/1111.6950

- [SVDMAT] Singular value decomposition and matrix reorderings in quantum information theory. Miszczak. Int. J. Mod. Phys. C 22, No. 9, 897 (2011). https://arxiv.org/abs/1011.1585

- [VECQO] Vectorization of quantum operations and its use. Gilchrist et al. arXiv:0911.2539 (2009). https://arxiv.org/abs/0911.2539

- [MATQO] On the Matrix Representation of Quantum Operations. Nambu et al. arXiv:0504091 (2005). https://arxiv.org/abs/quant-ph/0504091

- [DUAL] On duality between quantum maps and quantum states. Zyczkowski et al. Open Syst. Inf. Dyn. 11, 3 (2004). https://arxiv.org/abs/quant-ph/0401119